# Sanskrit Character Recognition using CNN

**Dataset:** Sanskrit Letter Dataset (DevDigitizer Project, BITS Pilani)
https://github.com/avadesh02/Sanskrit-letter-dataset

7702 images of Sanskrit letters segmented from real Sanskrit documents,
belonging to 602 classes, labelled with I-Trans transliteration.

We use a **small subset**: 20 letter classes with 25 images each = 500 images.

# Download the dataset

In [ ]:
!wget -q https://raw.githubusercontent.com/avadesh02/Sanskrit-letter-dataset/master/dev_letter_D.p
print("Dataset downloaded successfully!")

# Libraries

In [ ]:
import pickle
import collections
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

np.random.seed(42)
tf.random.set_seed(42)

# Load the dataset file

The dataset is stored as a pickle file. Each item has 3 parts:
1. the image array
2. the class index number
3. the English (I-Trans) label of the Sanskrit letter

In [ ]:
db = pickle.load(open("dev_letter_D.p", "rb"), encoding="latin1")

print("Total images in dataset:", len(db))
print("Image shape:", np.array(db[0][0]).shape)
print("Class index:", db[0][1])
print("Letter label:", db[0][2])

# Select a small subset of classes

In [ ]:
NUM_CLASSES      = 20   # number of Sanskrit letters to use
IMAGES_PER_CLASS = 25   # images taken per letter
IMG_SIZE         = 32   # image size (32 x 32)
EPOCHS           = 30

# count how many images each letter has, and keep only letter labels
counts = collections.Counter([item[2] for item in db])
all_letters = [c for c, n in counts.most_common() if c.isalpha()]

selected_classes = all_letters[:NUM_CLASSES]
print("Selected Sanskrit letters:")
print(selected_classes)

# Load and preprocess the images

In [ ]:
X = []
Y = []
per_class = collections.Counter()

for img, class_idx, label in db:
    if label in selected_classes and per_class[label] < IMAGES_PER_CLASS:
        per_class[label] += 1
        img = np.array(img, dtype=np.uint8)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # convert to grayscale
        gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))    # resize to 32 x 32
        X.append(gray)
        Y.append(label)

X = np.array(X)
Y = np.array(Y)

print("Data shape:", X.shape)
print("Labels shape:", Y.shape)

# Show some sample Sanskrit characters

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    idx = np.random.randint(0, len(X))
    plt.subplot(2, 5, i + 1)
    plt.imshow(X[idx], cmap='gray')
    plt.title(Y[idx], fontsize=10)
    plt.axis('off')
plt.suptitle("Sample Sanskrit Characters")
plt.tight_layout()
plt.show()

# Split into training and testing sets

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

# Normalize the pixels and reshape for CNN

In [ ]:
X_train = X_train / 255.0
X_test  = X_test / 255.0

# CNN needs 4D input: (samples, height, width, channels)
X_train = X_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
X_test  = X_test.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Encode the labels

In [ ]:
le = LabelEncoder()
Y_train_enc = le.fit_transform(Y_train)
Y_test_enc  = le.transform(Y_test)

Y_train_cat = to_categorical(Y_train_enc, NUM_CLASSES)

class_names = le.classes_
print("Classes:", class_names)

# Create the CNN model

In [ ]:
model = Sequential([
    Input((IMG_SIZE, IMG_SIZE, 1)),

    Conv2D(16, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

# Compile the model

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model

In [ ]:
history = model.fit(X_train, Y_train_cat,
                    epochs=EPOCHS,
                    batch_size=32,
                    validation_split=0.2,
                    verbose=1)

# Predict on the test set

In [ ]:
Y_pred = np.argmax(model.predict(X_test), axis=1)

# Calculate Accuracy

In [ ]:
accuracy = accuracy_score(Y_test_enc, Y_pred)
print("Test Accuracy:", accuracy)

# Classification Report (Precision, Recall, F1-score)

In [ ]:
print("Classification Report:")
print(classification_report(Y_test_enc, Y_pred,
                            target_names=class_names, zero_division=0))

# Confusion Matrix

In [ ]:
cm = confusion_matrix(Y_test_enc, Y_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Training vs Validation performance

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

# Visualize classification results

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    idx = np.random.randint(0, len(X_test))
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')

    actual = class_names[Y_test_enc[idx]]
    pred   = class_names[Y_pred[idx]]
    color  = 'green' if actual == pred else 'red'

    plt.title("A: " + actual + "\nP: " + pred, fontsize=9, color=color)
    plt.axis('off')

plt.suptitle("Actual (A) vs Predicted (P)")
plt.tight_layout()
plt.show()

# Predict a single unseen character image

Here we save one test image as a PNG file and then load it back,
to show how the model classifies a completely new image file.

In [ ]:
# save a test image as a png file
cv2.imwrite("test_char.png", (X_test[0].reshape(IMG_SIZE, IMG_SIZE) * 255).astype(np.uint8))

# now read that image and predict
img = cv2.imread("test_char.png", cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img = img / 255.0
img = img.reshape(1, IMG_SIZE, IMG_SIZE, 1)

prediction = model.predict(img)

print("Predicted Character:", class_names[np.argmax(prediction)])
print("Actual Character   :", class_names[Y_test_enc[0]])
print("Confidence         :", np.max(prediction))

# Save the trained model

In [ ]:
model.save("sanskrit_cnn_model.h5")
print("Model saved successfully!")